In [1]:
import json
import os
from pathlib import Path
from pprint import pprint
import numpy as np

from datasets import load_from_disk
from transformers import AutoModelForTokenClassification, AutoTokenizer,\
DataCollatorForTokenClassification, TrainingArguments, Trainer, EvalPrediction
import evaluate

import wandb
from colorama import Fore, Style

In [2]:
from transformers.models.deberta_v2.tokenization_deberta_v2 import DebertaV2Tokenizer
from transformers.models.deberta_v2.modeling_deberta_v2 import DebertaV2ForTokenClassification
from datasets import DatasetDict, Dataset

In [3]:
model_path = "microsoft/deberta-v3-base"

In [4]:
DATA_DIR = Path(os.getcwd()).parent / "data"
DATA_DIR.mkdir(exist_ok=True)
LABEL_INFO_PATH = DATA_DIR/"label_info.json"
DATASET_PATH = DATA_DIR/"cleaned_ai4privacy_300k_pii"

MODEL_DIR = Path(os.getcwd()).parent / "models"
MODEL_DIR.mkdir(exist_ok=True)
RUN_NAME = "pii_redaction_" + model_path.replace("/", "_") + "_v1"

In [5]:
dataset: DatasetDict = load_from_disk(DATASET_PATH) # type:ignore
print(f"dataset column names: {dataset["train"].column_names}\n\n")

dataset column names: ['source_text', 'privacy_mask']




In [6]:
# label info
with open(LABEL_INFO_PATH) as f:
    label_info: dict = json.load(f)
label2id = label_info["label2id"]
id2label = label_info["id2label"]

In [7]:

tokenizer: DebertaV2Tokenizer = AutoTokenizer.from_pretrained(model_path)
model: DebertaV2ForTokenClassification = AutoModelForTokenClassification.from_pretrained(
    model_path, num_labels=len(id2label), id2label=id2label, label2id=label2id)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task

In [8]:
print(type(tokenizer))
print(type(model))

<class 'transformers.models.deberta_v2.tokenization_deberta_v2.DebertaV2Tokenizer'>
<class 'transformers.models.deberta_v2.modeling_deberta_v2.DebertaV2ForTokenClassification'>


In [9]:
tokenizer.is_fast

True

In [10]:
example = dataset["train"][0]
print("dataset example:")
for item in example.items():
    print(item)

dataset example:
('source_text', 'Subject: Group Messaging for Admissions Process\n\nGood morning, everyone,\n\nI hope this message finds you well. As we continue our admissions processes, I would like to update you on the latest developments and key information. Please find below the timeline for our upcoming meetings:\n\n- wynqvrh053 - Meeting at 10:20am\n- luka.burg - Meeting at 21\n- qahil.wittauer - Meeting at quarter past 13\n- gholamhossein.ruschke - Meeting at 9:47 PM\n- pdmjrsyoz1460 ')
('privacy_mask', [{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}, {'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'}, {'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'}, {'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'}, {'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'}, {'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'}, {'value': 'gholamhossein.ruschke', 'start': 395, 'end': 416, 'la

In [11]:
ex_tokenized_with_offsets = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=False,
)
for item in ex_tokenized_with_offsets.items():
    print(item)

('input_ids', [11819, 294, 1543, 48850, 270, 30180, 8447, 1798, 1066, 261, 837, 261, 273, 875, 291, 1424, 4758, 274, 371, 260, 463, 301, 959, 316, 13656, 2422, 261, 273, 338, 334, 264, 1981, 274, 277, 262, 1120, 5908, 263, 896, 439, 260, 863, 433, 748, 262, 9808, 270, 316, 3181, 3315, 294, 341, 507, 23350, 4553, 27402, 1537, 62972, 341, 6754, 288, 466, 294, 1435, 1764, 341, 2531, 17910, 260, 10828, 341, 6754, 288, 1259, 341, 75407, 33442, 260, 55479, 34092, 341, 6754, 288, 1810, 695, 989, 341, 2905, 58588, 358, 41002, 268, 26266, 260, 8926, 61610, 341, 6754, 288, 712, 294, 5753, 2839, 341, 845, 407, 358, 44461, 268, 608, 9919, 2160, 3346])
('token_type_ids', [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [12]:
tokens = ex_tokenized_with_offsets.tokens()
print(tokens)

['▁Subject', ':', '▁Group', '▁Messaging', '▁for', '▁Admissions', '▁Process', '▁Good', '▁morning', ',', '▁everyone', ',', '▁I', '▁hope', '▁this', '▁message', '▁finds', '▁you', '▁well', '.', '▁As', '▁we', '▁continue', '▁our', '▁admissions', '▁processes', ',', '▁I', '▁would', '▁like', '▁to', '▁update', '▁you', '▁on', '▁the', '▁latest', '▁developments', '▁and', '▁key', '▁information', '.', '▁Please', '▁find', '▁below', '▁the', '▁timeline', '▁for', '▁our', '▁upcoming', '▁meetings', ':', '▁-', '▁', 'wyn', 'q', 'vr', 'h', '053', '▁-', '▁Meeting', '▁at', '▁10', ':', '20', 'am', '▁-', '▁l', 'uka', '.', 'burg', '▁-', '▁Meeting', '▁at', '▁21', '▁-', '▁qa', 'hil', '.', 'witt', 'auer', '▁-', '▁Meeting', '▁at', '▁quarter', '▁past', '▁13', '▁-', '▁g', 'hola', 'm', 'hos', 's', 'ein', '.', 'ru', 'schke', '▁-', '▁Meeting', '▁at', '▁9', ':', '47', '▁PM', '▁-', '▁p', 'd', 'm', 'jr', 's', 'y', 'oz', '14', '60']


In [13]:
word_ids = ex_tokenized_with_offsets.word_ids()
print(word_ids)

[0, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, 8, 9, 10, 11, 12, 13, 14, 15, 15, 16, 17, 18, 19, 20, 21, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 43, 44, 45, 45, 45, 45, 45, 45, 46, 47, 48, 49, 49, 49, 49, 50, 51, 51, 51, 51, 52, 53, 54, 55, 56, 57, 57, 57, 57, 57, 58, 59, 60, 61, 62, 63, 64, 65, 65, 65, 65, 65, 65, 65, 65, 65, 66, 67, 68, 69, 69, 69, 70, 71, 72, 72, 72, 72, 72, 72, 72, 72, 72]


In [14]:
pprint(example["privacy_mask"], sort_dicts=False)

[{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'},
 {'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'},
 {'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'},
 {'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'},
 {'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'},
 {'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'},
 {'value': 'gholamhossein.ruschke',
  'start': 395,
  'end': 416,
  'label': 'USERNAME'},
 {'value': '9:47 PM', 'start': 430, 'end': 437, 'label': 'TIME'},
 {'value': 'pdmjrsyoz1460', 'start': 440, 'end': 453, 'label': 'USERNAME'}]


In [15]:
def get_ner_labels(batch: dict[str, list]) -> dict[str, list[list[str]]]:
    token_offsets = tokenizer(
        batch["source_text"],
        return_offsets_mapping=True,
        add_special_tokens=False,
    )["offset_mapping"]

    batch_ner_labels: list[list[str]] = []
    # loop through the batch to get the privacy masks for every sequence
    for i, row_masks in enumerate(batch["privacy_mask"]):
        row_ner_labels = []
        # loop through the token_offsets of the current sequence
        for offset in token_offsets[i]:
            # if add_special_tokens is true skip over special tokens (offset with (0,0))
            if offset == (0, 0): 
                row_ner_labels.append("O")
                continue
            # label is "O" unless privacy label is found
            label = "O" 
            # loop through masks to find label
            for privacy_mask in row_masks:
                # if statement is switched to check if any character of the token falls within the mask
                if offset[1] > privacy_mask["start"] and offset[0] < privacy_mask["end"]:
                    label = privacy_mask["label"]
                    # switch if statement to also include less than
                    if offset[0] <= privacy_mask["start"]:
                        label = "B-" + label
                    else:
                        label = "I-" + label
                    
                    break # break out of for loop when/if label is found
            row_ner_labels.append(label)
            
        batch_ner_labels.append(row_ner_labels)
    
    return {"ner_tags": batch_ner_labels}

In [16]:
dataset = dataset.map(get_ner_labels, batched=True, batch_size=20_000)
pprint(dataset["train"].features)

Map:   0%|          | 0/29908 [00:00<?, ? examples/s]

Map:   0%|          | 0/3973 [00:00<?, ? examples/s]

Map:   0%|          | 0/3973 [00:00<?, ? examples/s]

{'ner_tags': List(Value('string')),
 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'source_text': Value('string')}


In [17]:
example = dataset["train"][0]
example_encodings = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=False,
)
line1 = ""
line2 = ""
line3 = ""
line4 = ""
for word, label, word_id, offset in zip(example_encodings.tokens(), example["ner_tags"], example_encodings.word_ids(), example_encodings["offset_mapping"]):
    full_label = label
    word_str = str(word_id)
    offset = str(offset)
    max_length = max(len(word), len(full_label), len(word_str), len(offset))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += full_label + " " * (max_length - len(full_label) + 1)
    line3 += word_str + " " * (max_length - len(word_str) + 1)
    line4 += offset + " " * (max_length - len(offset) + 1)
print(line1)
print(line2)
print(line3)
print(line4)

▁Subject :      ▁Group  ▁Messaging ▁for     ▁Admissions ▁Process ▁Good    ▁morning ,        ▁everyone ,        ▁I       ▁hope    ▁this    ▁message ▁finds   ▁you      ▁well      .          ▁As        ▁we        ▁continue  ▁our       ▁admissions ▁processes ,          ▁I         ▁would     ▁like      ▁to        ▁update    ▁you       ▁on        ▁the       ▁latest    ▁developments ▁and       ▁key       ▁information .          ▁Please    ▁find      ▁below     ▁the       ▁timeline  ▁for       ▁our       ▁upcoming  ▁meetings  :          ▁-         ▁          wyn        q          vr         h          053        ▁-         ▁Meeting   ▁at        ▁10        :          20         am         ▁-         ▁l         uka        .          burg       ▁-         ▁Meeting   ▁at        ▁21        ▁-         ▁qa        hil        .          witt       auer       ▁-         ▁Meeting   ▁at        ▁quarter   ▁past      ▁13        ▁-         ▁g         hola       m          hos        s          ein        .  

In [ ]:
# token include whitespace
print(example["source_text"][310:313].replace(" ", "_"))

_10


In [34]:
text = "Call John Smith at 555-1234"
enc = tokenizer(text, return_offsets_mapping=True)
for tok, off in zip(enc.tokens(), enc["offset_mapping"]):
    print(f"{tok:20s} {off}  →  '{text[off[0]:off[1]]}'")

[CLS]                (0, 0)  →  ''
▁Call                (0, 4)  →  'Call'
▁John                (4, 9)  →  ' John'
▁Smith               (9, 15)  →  ' Smith'
▁at                  (15, 18)  →  ' at'
▁555                 (18, 22)  →  ' 555'
-                    (22, 23)  →  '-'
1234                 (23, 27)  →  '1234'
[SEP]                (0, 0)  →  ''


In [29]:
for key, value in example.items():
    print(f"{key}:{len(value)}")

token_offsets = example_encodings["offset_mapping"]   
print(token_offsets[0], token_offsets[-1])

source_text:454
privacy_mask:9
ner_tags:113
(0, 7) (451, 453)


In [17]:
def tokenize_and_align_labels_single(example: dict[str, list]):
    tokenized_input = tokenizer(
        example["source_text"],
        truncation=True,
        add_special_tokens=False
    )
    labels = []
    word_ids = tokenized_input.word_ids()
    for i, tag in enumerate(example["ner_tags"]):
        previous_word_idx = None

        word_idx = word_ids[i]
        
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            # in case there is a label not in label2id in becomes 0
            labels.append(label2id.get(tag, 0))
        else:
            labels.append(-100)
        previous_word_idx = word_idx
            
    tokenized_input["labels"] = labels
    return tokenized_input
            

In [18]:
dataset = dataset.map(tokenize_and_align_labels_single, batched=False)

Map:   0%|          | 0/29908 [00:00<?, ? examples/s]

Map:   0%|          | 0/3973 [00:00<?, ? examples/s]

Map:   0%|          | 0/3973 [00:00<?, ? examples/s]

In [19]:
pprint(dataset["train"].features)

{'attention_mask': List(Value('int8')),
 'input_ids': List(Value('int32')),
 'labels': List(Value('int64')),
 'ner_tags': List(Value('string')),
 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'source_text': Value('string'),
 'token_type_ids': List(Value('int8'))}


In [ ]:
print(f"{len(example["ner_tags"])=}")
print(f"{len(example["input_ids"])=}")

len(example["ner_tags"])=113
len(example["input_ids"])=113


In [22]:
ex_wi = tokenizer(
    example["source_text"],
    truncation=True,
    add_special_tokens=False
).word_ids()
masks = example["privacy_mask"]

token_offsets = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=False,
)["offset_mapping"]

id2label = {i: label for label, i in label2id.items()}
tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])

In [23]:
for mask in masks:
    print(Fore.BLUE + f"{mask}" + Style.RESET_ALL)    
for label, wi, offset in zip(example["labels"], ex_wi, token_offsets):
    if label != -100 and label != 0:
        token = tokens[wi] #type:ignore
        value = f"label={id2label[label]}, word_id={wi}, {token=}, {offset=}"
        print(value)
        
        masks_str = []
        for mask in masks:
            if offset[0] >= mask["start"] and offset[1] <= mask["end"]:
                masks_str.append(f"in {mask}")
                
        if masks_str:
            print(Fore.GREEN + f"{masks_str}")
            print(Style.RESET_ALL + "", end="")
        else:
            print(Fore.RED + f" {offset} not in masks" )
            print(Style.RESET_ALL + "", end="")

{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}
{'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'}
{'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'}
{'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'}
{'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'}
{'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'}
{'value': 'gholamhossein.ruschke', 'start': 395, 'end': 416, 'label': 'USERNAME'}
{'value': '9:47 PM', 'start': 430, 'end': 437, 'label': 'TIME'}
{'value': 'pdmjrsyoz1460', 'start': 440, 'end': 453, 'label': 'USERNAME'}
label=B-USERNAME, word_id=45, token='▁timeline', offset=(287, 290)
["in {'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}"]
label=I-USERNAME, word_id=45, token='▁timeline', offset=(290, 291)
["in {'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}"]
label=I-USERNAME, word_id=45, token='▁timeline', offset=(291, 293)
["in {'value

In [21]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

seqeval = evaluate.load("seqeval")

In [22]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/zelluzy/.netrc.
wandb: Currently logged in as: bengid (bengid-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [23]:
def compute_metrics(p: EvalPrediction) -> dict[str, float]:
    predictions, label_ids = p.predictions, p.label_ids
    predictions = np.argmax(predictions, axis=-1)
    
    true_preds = [
        [id2label[str(pred)] for pred, tgt_id in zip(preds_row, labels_row) if tgt_id != -100]
        for preds_row, labels_row in zip(predictions, label_ids)
    ]
    
    true_labels =[
        [id2label[str(tgt_id)] for tgt_id in labels_row if tgt_id != -100]
        for labels_row in label_ids
    ]
    
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    
    flat = {}
    if results is not None:
        flat.update({
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"],
        })
        
        for entity, scores in results.items():
            if isinstance(scores, dict): # filter out scores for individual labels
                flat[f"{entity}_f1"] = scores["f1"]
                flat[f"{entity}_support"] = scores["number"]
    
    return flat

In [22]:
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    report_to="wandb",
    run_name=RUN_NAME,
    learning_rate=2e-5, # low learning_rate
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

NameError: name 'data_collator' is not defined

In [25]:
wandb.init(project="pii-redaction")
trainer_output = trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Bod F1,Bod Support,Building F1,Building Support,City F1,City Support,Country F1,Country Support,Date F1,Date Support,Driverlicense F1,Driverlicense Support,Email F1,Email Support,Geocoord F1,Geocoord Support,Givenname1 F1,Givenname1 Support,Givenname2 F1,Givenname2 Support,Idcard F1,Idcard Support,Ip F1,Ip Support,Lastname1 F1,Lastname1 Support,Lastname2 F1,Lastname2 Support,Lastname3 F1,Lastname3 Support,Pass F1,Pass Support,Passport F1,Passport Support,Postcode F1,Postcode Support,Secaddress F1,Secaddress Support,Sex F1,Sex Support,Socialnumber F1,Socialnumber Support,State F1,State Support,Street F1,Street Support,Tel F1,Tel Support,Time F1,Time Support,Title F1,Title Support,Username F1,Username Support
1,0.000000,nan,0.000000,0.000000,0.000000,0.873834,0.000000,480,0.000000,1,0.000000,163,0.000000,30,0.000000,353,0.000000,435,0.000000,486,0.000000,28,0.000000,192,0.000000,44,0.000000,518,0.000000,342,0.000000,271,0.000000,53,0.000000,15,0.000000,277,0.000000,455,0.000000,142,0.000000,59,0.000000,82,0.000000,514,0.000000,18,0.000000,283,0.000000,366,0.000000,487,0.000000,70,0.000000,554


/home/zelluzy/Desktop/pii_redaction_api/.venv/lib/python3.14/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/zelluzy/Desktop/pii_redaction_api/.venv/lib/python3.14/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [26]:
trainer_output

TrainOutput(global_step=1870, training_loss=0.0, metrics={'train_runtime': 153.9911, 'train_samples_per_second': 194.219, 'train_steps_per_second': 12.144, 'total_flos': 2865124090141800.0, 'train_loss': 0.0, 'epoch': 1.0})

In [21]:
out = trainer.predict(dataset["test"])
preds = np.argmax(out.predictions, axis=-1)
print(preds)
print(Fore.CYAN + "")
pprint(out.metrics)

NameError: name 'trainer' is not defined

In [29]:
wandb.finish()

eval/BOD_f1,▁
eval/BOD_support,▁
eval/BUILDING_f1,▁
eval/BUILDING_support,▁
eval/CITY_f1,▁
eval/CITY_support,▁
eval/COUNTRY_f1,▁
eval/COUNTRY_support,▁
eval/DATE_f1,▁
eval/DATE_support,▁
+117,...
